# PadelVision — 09 Retrain the ball detector on a new court (Kaggle)

On GX010012 (19 min match, another club) the current ball model (`ball-detector` v1) finds the real ball on only
about a third of the frames, so most of the play is lost. This notebook fine-tunes it with this court's footage.

1. **Labels without hand-labelling** from this video's **train blocks** (20 s blocks: odd = test, held out with a
   2 s margin): motion-based pseudo-labels (`ball-bootstrap`) + the v1 model's confidently tracked ball.
2. **Training data:** those + Test_video's labels (if its video is attached) + Roboflow Universe padel datasets
   (Kaggle secret `ROBOFLOW_API_KEY`).
3. **Fine-tune** from the v1 weights (not from COCO).
4. **Compare v1 and v2 on the held-out test blocks** (no labels needed): how often the ball in play is found,
   hits, rallies.
5. **Eval pack:** 200 test-block frames to label locally (`padelvision label-ball`), for real recall/precision of
   both models.

**Setup:** GPU **T4 x2** (or P100), **Internet on**, secret `ROBOFLOW_API_KEY` (Add-ons → Secrets; optional).
Optional datasets: notebook 08's `run_cache.zip` for this video (skips ~30 min of detection) and the Test_video
dataset. Total ~3–4 h: use **Save Version → Save & Run All (Commit)**. Downloads at the end: `ball_v2.zip`.

## 0. Settings

In [ ]:
DRIVE_URL = "https://drive.google.com/file/d/1PfNolYj0SmwjrZ1MiMvwEVXTSbictTDH/view?usp=sharing"
VIDEO_PATH = ""          # or the match video attached as a Kaggle dataset
TARGET_WIDTH, TARGET_FPS = 1280, 30
MAX_MINUTES = None       # keep None: training wants the whole match

EPOCHS = 30              # fine-tuning from v1 converges faster than training from COCO
IMGSZ = 1280
BATCH = 8
BLOCK_S = 20.0           # train/test blocks
USE_UNIVERSE = True      # needs the ROBOFLOW_API_KEY secret; otherwise skipped
UNIVERSE = [("padel-analysis-eulgl", "padel-ball-detection-nazbq", 4),
            ("padel-ll7pp", "padel-dataset", 1)]
N_EVAL_TRACKED, N_EVAL_RANDOM = 100, 100   # eval pack frames (test blocks)
SEED = 0
REBUILD = False          # True = rebuild the dataset / restart training in this session

## 1. Setup

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]" gdown roboflow

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

## 2. The match video (downloaded and converted as in notebook 08)

In [ ]:
import json, shutil, time
from pathlib import Path

VIDEO_EXT = (".mp4", ".mov", ".mkv", ".avi", ".m4v")
TMP = Path("/tmp/padel")
TMP.mkdir(parents=True, exist_ok=True)
print(f"free disk in /tmp: {shutil.disk_usage('/tmp').free / 1e9:.0f} GB")

def downloaded():
    return sorted(p for p in TMP.iterdir() if p.suffix.lower() in VIDEO_EXT and not p.name.startswith("work_"))

if VIDEO_PATH:
    SRC = Path(VIDEO_PATH)
elif downloaded():
    SRC = downloaded()[0]
    print("Already downloaded:", SRC)
else:
    assert shutil.disk_usage("/tmp").free > 20e9, "Need ~20 GB free in /tmp (the video + a converted copy)"
    t0 = time.time()
    !gdown --fuzzy "{DRIVE_URL}" -O {TMP}/
    assert downloaded(), ("Download failed. Check the Drive file is shared as 'Anyone with the link'. If Google "
                          "reports the download quota exceeded, wait and retry, or upload the video as a Kaggle "
                          "Dataset and set VIDEO_PATH.")
    SRC = downloaded()[0]
    print(f"Downloaded {SRC.name}: {SRC.stat().st_size / 1e9:.1f} GB in {(time.time() - t0) / 60:.1f} min")

STEM = SRC.stem
from padelvision.video import probe
src_info = probe(SRC)
print(f"{SRC.name}: {src_info.width}x{src_info.height} @ {src_info.fps:.2f} fps, "
      f"{src_info.duration_s / 60:.1f} min, {src_info.frame_count} frames")

In [ ]:
w, h, fps = src_info.width, src_info.height, src_info.fps
out_w = TARGET_WIDTH
out_h = round(h * TARGET_WIDTH / w / 2) * 2
lower_fps = fps > TARGET_FPS + 0.5
if (w, h) == (out_w, out_h) and not lower_fps and not MAX_MINUTES:
    VIDEO = SRC
    print("Already in the working format:", VIDEO)
else:
    VIDEO = TMP / f"work_{STEM}{'_' + str(MAX_MINUTES) + 'min' if MAX_MINUTES else ''}.mp4"
    if not VIDEO.exists():
        t0 = time.time()
        vf = f"scale={out_w}:{out_h}:flags=area" + (f",fps={TARGET_FPS}" if lower_fps else "")
        dur = f"-t {MAX_MINUTES * 60}" if MAX_MINUTES else ""
        print(f"Converting {src_info.duration_s / 60:.0f} min of {w}x{h} video to {out_w}x{out_h}: "
              "often 15-45 min for 4K; no progress output until it's done")
        !ffmpeg -hide_banner -loglevel error -hwaccel auto -i "{SRC}" {dur} -vf "{vf}" -an -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p "{VIDEO}"
        print(f"\nConverted in {(time.time() - t0) / 60:.1f} min")
info = probe(VIDEO)
print(f"Working video: {info.width}x{info.height} @ {info.fps:.2f} fps, {info.duration_s / 60:.1f} min, "
      f"{info.frame_count} frames")
assert info.frame_count > 0, "The converted video can't be read"

## 3. Run folder with the v1 results

Uses notebook 08's `run_cache.zip` if attached (same video: frame count checked), else runs player detection and v1
ball detection here (~30 min on a T4). Stages 3–7 are then refreshed with the current code: that is the v1 baseline.

In [ ]:
import shutil
from padelvision.court import CourtCalibration

RUN = Path(WORK) / "runs" / f"{STEM}_v1"
cal_file = Path(REPO) / "ml" / "calibrations" / f"{STEM}.json"
assert cal_file.exists(), f"No calibration {cal_file}: calibrate this video first (notebook 08, section 4)"
COURT_JSON = f"{WORK}/court.json"
CourtCalibration.load(cal_file).save(COURT_JSON)

def cached_runs():
    if not Path("/kaggle/input").exists():
        return []
    out = []
    for p in Path("/kaggle/input").rglob("ball_detections.parquet"):
        d = p.parent
        if (d / "detections.parquet").exists() and (d / "video.json").exists():
            v = json.loads((d / "video.json").read_text())
            if v["frame_count"] == info.frame_count and (v["width"], v["height"]) == (info.width, info.height):
                out.append(d)
    return out

if not (RUN / "ball_detections.parquet").exists():
    found = cached_runs()
    if found:
        print("Using cached run", found[0])
        shutil.copytree(found[0], RUN, dirs_exist_ok=True)
    else:
        print("No cached run of this video: detecting players and the ball with v1 (~30 min)")
        !cd {REPO} && python -m padelvision.cli analyze "{VIDEO}" --court {COURT_JSON} --out {RUN}
        !cd {REPO} && python -m padelvision.cli ball-track "{VIDEO}" --run {RUN}
!cd {REPO} && python -m padelvision.cli restats {RUN} --court {COURT_JSON}
!cd {REPO} && python -m padelvision.cli ball-retrack {RUN}
!cd {REPO} && python -m padelvision.cli events {RUN}

## 4. Motion-based pseudo-labels (`ball-bootstrap`, CPU, ~15–25 min)

Small moving blobs outside player boxes, linked into smooth fast flights. Independent of the v1 model, so it can find
balls v1 misses.

In [ ]:
t0 = time.time()
if not (RUN / "ball_pseudo.parquet").exists():
    !cd {REPO} && python -m padelvision.cli ball-bootstrap "{VIDEO}" --run {RUN}
print(f"bootstrap: {(time.time() - t0) / 60:.1f} min")

## 5. Training dataset

In [ ]:
import cv2
import numpy as np
import pandas as pd
from padelvision.ball.dataset import (ball_size_by_row, count, import_yolo_dataset, is_test_frame,
                                      is_train_frame, merge_ball_labels, write_data_yaml,
                                      write_video_frames)

fps = info.fps
DS = Path(f"{WORK}/ball_ds_v2")
pseudo = pd.read_parquet(RUN / "ball_pseudo.parquet")
ball_v1 = pd.read_parquet(RUN / "ball.parquet")
size_of = ball_size_by_row(ball_v1)
pseudo = pseudo.assign(size=size_of(pseudo.v))
tracked = ball_v1[(ball_v1.state == "detected") & ball_v1["size"].notna()]
own = merge_ball_labels([tracked, pseudo])            # v1 tracked ball first (it has real sizes)
own = own[is_train_frame(own.frame, fps, BLOCK_S)]
frames = sorted(own.frame.unique())
rng = np.random.default_rng(SEED)
val = set(rng.choice(frames, max(1, len(frames) // 10), replace=False))
print(f"{STEM} train blocks: {len(frames)} labelled frames, {len(own)} balls "
      f"({len(tracked[is_train_frame(tracked.frame, fps, BLOCK_S)])} v1-tracked, "
      f"{len(pseudo[is_train_frame(pseudo.frame, fps, BLOCK_S)])} motion pseudo-labels before merging)")

READY = (DS / "data.yaml").exists() and (DS / "sources.json").exists() and not REBUILD
if READY:
    SOURCES = json.loads((DS / "sources.json").read_text())
    print("reusing the dataset built earlier in this session:", count(DS))
else:
    shutil.rmtree(DS, ignore_errors=True)
    n_tr = write_video_frames(VIDEO, own[~own.frame.isin(val)], DS, "train", STEM)
    n_va = write_video_frames(VIDEO, own[own.frame.isin(val)], DS, "val", STEM)
    SOURCES = [{"source": f"{STEM} (train blocks): v1-tracked ball + motion pseudo-labels",
                "images": n_tr + n_va, "balls": len(own)}]

    # Test_video's labels (its train blocks), if that video is attached: keeps v1's court in the mix.
    tv = sorted(Path("/kaggle/input").rglob("Test_video.mp4")) if Path("/kaggle/input").exists() else []
    if tv:
        tv_fps = cv2.VideoCapture(str(tv[0])).get(cv2.CAP_PROP_FPS)
        labels = pd.read_csv(f"{REPO}/ml/eval/labels/Test_video_ball.csv")
        tp = pd.read_csv(f"{REPO}/ml/datasets/Test_video_ball_pseudo.csv")
        tp = tp[is_train_frame(tp.frame, tv_fps, BLOCK_S) & ~tp.frame.isin(set(labels.frame))]
        conf = labels[(labels.status == "ball") & is_train_frame(labels.frame, tv_fps, BLOCK_S)]
        tv_boxes = pd.concat([tp[["frame", "u", "v", "size"]], conf.assign(size=16.0)[["frame", "u", "v", "size"]]])
        n = write_video_frames(tv[0], tv_boxes, DS, "train", "Test_video")
        SOURCES.append({"source": "Test_video.mp4 (train blocks)", "images": n, "balls": len(tv_boxes)})
    else:
        print("Test_video not attached: skipping its labels")

    rf_key = None
    if USE_UNIVERSE:
        try:
            from kaggle_secrets import UserSecretsClient
            rf_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        except Exception:
            rf_key = os.environ.get("ROBOFLOW_API_KEY")
    if rf_key:
        from roboflow import Roboflow
        rf = Roboflow(api_key=rf_key)
        for ws, proj, ver in UNIVERSE:
            try:
                dl = rf.workspace(ws).project(proj).version(ver).download(
                    "yolov8", location=f"{WORK}/universe/{proj}", overwrite=True)
                st = import_yolo_dataset(dl.location, DS, prefix=proj)
                st["id"] = f"{ws}/{proj}/{ver}"
                SOURCES.append(st)
                print(f"{proj}: {st['images']} images, {st['balls']} balls")
            except Exception as e:
                print(f"{proj} failed:", repr(e)[:200])
    else:
        print("No ROBOFLOW_API_KEY: training without the Universe datasets")
DATA = DS / "data.yaml" if READY else write_data_yaml(DS, SOURCES)
print("dataset:", count(DS))

## 6. Fine-tune from v1

Resumes from the last epoch if this session was interrupted (re-run from the top; `REBUILD = False`).

In [ ]:
import torch
from ultralytics import YOLO
from padelvision.models import weights_path

LAST = Path(f"{WORK}/train/ball_v2/weights/last.pt")
devices = [0, 1] if torch.cuda.device_count() > 1 else 0
t0 = time.time()
if LAST.exists() and not REBUILD:
    try:
        YOLO(str(LAST)).train(resume=True)
    except Exception as e:
        print("not resuming:", repr(e)[:200])
else:
    YOLO(str(weights_path("ball-detector"))).train(
        data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=devices,
        project=f"{WORK}/train", name="ball_v2", exist_ok=True, seed=SEED,
        patience=10, cos_lr=True, close_mosaic=5, single_cls=True,
        degrees=0.0, fliplr=0.5, scale=0.3, workers=4, plots=True,
    )
BEST = Path(f"{WORK}/train/ball_v2/weights/best.pt")
print(f"training {(time.time() - t0) / 60:.0f} min; best weights:", BEST, BEST.exists())

## 7. v2 on the whole match (~15 min), then tracking and events

In [ ]:
from padelvision.models.ball_detectors import YoloBall

RUN2 = Path(WORK) / "runs" / f"{STEM}_v2"
RUN2.mkdir(parents=True, exist_ok=True)
for name in ["video.json", "court.json", "detections.parquet", "detections.json", "players.parquet",
             "tracks.parquet", "stats.json"]:
    shutil.copy(RUN / name, RUN2 / name)
if not (RUN2 / "ball_detections.parquet").exists():
    t0 = time.time()
    dets = YoloBall(BEST, classes=[0], imgsz=IMGSZ, conf=0.05).detect_video(VIDEO)
    dets.to_parquet(RUN2 / "ball_detections.parquet")
    (RUN2 / "ball_detections.json").write_text(json.dumps(
        {"model": "ball-detector v2 (notebook 09)", "max_frames": None, "imgsz": IMGSZ, "conf": 0.05,
         "columns": "frame,u,v,conf,size"}))
    print(f"{len(dets)} detections in {(time.time() - t0) / 60:.1f} min")
!cd {REPO} && python -m padelvision.cli ball-retrack {RUN2}
!cd {REPO} && python -m padelvision.cli events {RUN2}

## 8. v1 vs v2 on the held-out test blocks

No labels needed: how often the ball in play is found, and what that does to hits and rallies. Only test-block numbers
are fair (v2 was trained on the train blocks, partly with v1's own output).

In [ ]:
def summary_for(run: Path) -> dict:
    ball = pd.read_parquet(run / "ball.parquet")
    ev = pd.read_parquet(run / "events.parquet")
    r = json.loads((run / "rallies.json").read_text())
    rallies = r["rallies"]
    test = is_test_frame(np.arange(info.frame_count), fps, BLOCK_S)
    in_test = lambda f: is_test_frame(f, fps, BLOCK_S)  # noqa: E731
    test_rallies = [x for x in rallies if in_test(x["start_frame"]) and in_test(x["end_frame"])]
    return {
        "ball found (test blocks)": f"{in_test(ball.frame).sum() / test.sum():.0%}",
        "ball found (all)": f"{len(ball) / info.frame_count:.0%}",
        "hits (test blocks)": int(((ev.kind == "hit") & in_test(ev.frame)).sum()),
        "bounces (test blocks)": int(((ev.kind == "bounce") & in_test(ev.frame)).sum()),
        "rallies inside test blocks": len(test_rallies),
        "rallies (all)": len(rallies),
        "rally time s (all)": round(sum(x["duration_s"] for x in rallies)),
        "hits per rally (all)": round(np.mean([x["hits"] for x in rallies]), 1) if rallies else 0,
    }

COMPARE = pd.DataFrame({"v1": summary_for(RUN), "v2": summary_for(RUN2)})
display(COMPARE)

## 9. Eval pack (label locally)

200 test-block frames during play: 100 where v1 or v2 tracked a ball, 100 random. The ring proposes v2's ball; label
with `padelvision label-ball data/ball_eval_v2` (Y = ring on the ball, click = the ball is elsewhere, N = no ball in
play visible). Both models' detections on these frames are in the zip, so they can be scored locally afterwards.

In [ ]:
from padelvision.ball.bootstrap import write_review_pack

ball_v2 = pd.read_parquet(RUN2 / "ball.parquet")
players = pd.read_parquet(RUN / "players.parquet")
in_play = players.groupby("frame").size()
play = np.array([f for f in in_play.index[in_play >= 2] if 0 < f < info.frame_count - 1])
play = play[is_test_frame(play, fps, BLOCK_S)]
tracked = np.intersect1d(play, np.union1d(ball_v1[ball_v1.state == "detected"].frame,
                                          ball_v2[ball_v2.state == "detected"].frame))
rng = np.random.default_rng(SEED)
pick_t = rng.choice(tracked, min(N_EVAL_TRACKED, len(tracked)), replace=False)
pick_r = rng.choice(np.setdiff1d(play, pick_t), min(N_EVAL_RANDOM, len(play)), replace=False)
dets_v2 = pd.read_parquet(RUN2 / "ball_detections.parquet")
top_v2 = dets_v2.sort_values("conf", ascending=False).drop_duplicates("frame").set_index("frame")
b2 = ball_v2.set_index("frame")
items = []
for f, source in [*((f, "tracked") for f in pick_t), *((f, "random") for f in pick_r)]:
    f = int(f)
    prop = (b2.loc[f, ["u", "v"]].tolist() if f in b2.index
            else top_v2.loc[f, ["u", "v"]].tolist() if f in top_v2.index else None)
    items.append({"frame": f, "t": f / fps, "source": source, "proposal": prop})
EVAL = Path(WORK) / "ball_v2" / "ball_eval_v2"
shutil.rmtree(EVAL.parent, ignore_errors=True)
EVAL.mkdir(parents=True)
write_review_pack(VIDEO, items, EVAL)
frames_set = {it["frame"] for it in items}
dets_v1 = pd.read_parquet(RUN / "ball_detections.parquet")
dets_v1[dets_v1.frame.isin(frames_set)].to_parquet(EVAL.parent / "eval_dets_v1.parquet")
dets_v2[dets_v2.frame.isin(frames_set)].to_parquet(EVAL.parent / "eval_dets_v2.parquet")
print(f"eval pack: {len(items)} frames ({len(pick_t)} tracked, {len(pick_r)} random)")

## 10. Save

In [ ]:
import hashlib
from IPython.display import FileLink

OUT = EVAL.parent
shutil.copy(BEST, OUT / "padel_ball_yolo11s_v2.pt")
sha = hashlib.sha256((OUT / "padel_ball_yolo11s_v2.pt").read_bytes()).hexdigest()
for extra in ["results.csv", "results.png", "args.yaml"]:
    p = Path(f"{WORK}/train/ball_v2/{extra}")
    if p.exists():
        shutil.copy(p, OUT / f"train_{extra}")
COMPARE.to_csv(OUT / "compare_v1_v2.csv")
(OUT / "model_card.json").write_text(json.dumps({
    "file": "padel_ball_yolo11s_v2.pt", "sha256": sha, "base": "ball-detector v1 (padel_ball_yolo11s.pt)",
    "imgsz": IMGSZ, "epochs": EPOCHS, "block_s": BLOCK_S, "sources": SOURCES,
    "compare_test_blocks": COMPARE.to_dict(),
}, indent=1, default=str))
cache = OUT / "run_v2"
cache.mkdir(exist_ok=True)
for name in ["video.json", "court.json", "stats.json", "players.parquet", "tracks.parquet",
             "ball_detections.parquet", "ball_detections.json", "ball.parquet", "ball_stats.json",
             "events.parquet", "rallies.json"]:
    if (RUN2 / name).exists():
        shutil.copy(RUN2 / name, cache / name)
print("sha256:", sha)
ZIP = shutil.make_archive(f"{WORK}/ball_v2", "zip", OUT.parent, OUT.name)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.0f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()))